In [1]:
import tensorflow as tf

# Local dataset paths
base_dir = "./dataset-kaggle"
train_dir = f"{base_dir}/train"
val_dir = f"{base_dir}/val"
test_dir = f"{base_dir}/test"

train_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, seed=123, image_size=(128, 128), batch_size=64
)

val_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir, seed=123, image_size=(128, 128), batch_size=64
)

test_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir, seed=123, image_size=(128, 128), batch_size=64, shuffle=False
)

c:\Users\shrey\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\shrey\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\shrey\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framewo

Found 5778 files belonging to 5 classes.
Found 826 files belonging to 5 classes.
Found 1656 files belonging to 5 classes.


In [6]:
import cv2
import numpy as np

def apply_clahe(img):
    img = img.numpy().astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(gray)
    # Convert back to 3 channels for base models like ResNet50
    cl_rgb = cv2.cvtColor(cl, cv2.COLOR_GRAY2RGB)
    return tf.cast(cl_rgb, tf.float32)

def to_ordinal(label):
    # Converts label to multi-hot ordinal vector (e.g. 3 -> [1, 1, 1, 0])
    label = tf.cast(label, tf.float32)
    thresholds = tf.constant([0.0, 1.0, 2.0, 3.0], dtype=tf.float32)
    return tf.cast(label > thresholds, tf.float32)

def process_data(img, label):
    # Wrap cv2 function for tf pipeline
    img = tf.py_function(apply_clahe, [img], tf.float32)
    img.set_shape([128, 128, 3])
    label = to_ordinal(label)
    return img, label

# Unbatch to process single images, then re-batch them for the model
batch_size = 64

train_ds = train_ds_raw.unbatch().map(process_data, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size)
val_ds = val_ds_raw.unbatch().map(process_data, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size)
test_ds = test_ds_raw.unbatch().map(process_data, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size)

In [7]:
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    input_shape=(128, 128, 3),
    include_top=False
)
base_model.trainable = True

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    # 4 outputs with Sigmoid for ordinal regression instead of 5 with Softmax
    tf.keras.layers.Dense(4, activation='sigmoid')  
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

def get_callbacks(model_name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"{model_name}.keras",
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
    ]

# Train using your existing callbacks
epochs = 50
callbacks = get_callbacks('ResNet50_Ordinal')
history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks)

Epoch 1/50
     91/Unknown 251s 3s/step - accuracy: 0.7292 - loss: 0.4991

c:\Users\shrey\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


91/91 ━━━━━━━━━━━━━━━━━━━━ 263s 3s/step - accuracy: 0.7305 - loss: 0.4980 - val_accuracy: 0.9734 - val_loss: 0.4125 - learning_rate: 1.0000e-04
Epoch 2/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 232s 3s/step - accuracy: 0.9180 - loss: 0.2652 - val_accuracy: 0.8668 - val_loss: 0.4461 - learning_rate: 1.0000e-04
Epoch 3/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 189s 2s/step - accuracy: 0.9223 - loss: 0.1433 - val_accuracy: 0.8450 - val_loss: 0.7111 - learning_rate: 1.0000e-04
Epoch 4/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 177s 2s/step - accuracy: 0.9052 - loss: 0.0693 - val_accuracy: 0.7324 - val_loss: 0.6923 - learning_rate: 5.0000e-05
Epoch 5/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 176s 2s/step - accuracy: 0.8672 - loss: 0.0271 - val_accuracy: 0.7058 - val_loss: 0.5874 - learning_rate: 5.0000e-05
Epoch 6/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 180s 2s/step - accuracy: 0.7884 - loss: 0.0129 - val_accuracy: 0.8087 - val_loss: 0.5425 - learning_rate: 2.5000e-05


In [8]:
from sklearn.metrics import mean_absolute_error, cohen_kappa_score

y_pred_prob = model.predict(test_ds)
# Threshold probabilities at 0.5 to get binary values, then sum across axis to get the severity grade [0, 1, 2, 3, 4]
predicted_categories = np.sum(y_pred_prob > 0.5, axis=1)

# Extract integer true labels from raw dataset
true_categories = np.concatenate([y for x, y in test_ds_raw], axis=0)

mae = mean_absolute_error(true_categories, predicted_categories)
kappa = cohen_kappa_score(true_categories, predicted_categories, weights='quadratic')

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Quadratic Weighted Kappa: {kappa:.4f}")

26/26 ━━━━━━━━━━━━━━━━━━━━ 14s 508ms/step
Mean Absolute Error (MAE): 0.7850
Quadratic Weighted Kappa: 0.4871
